# Phase 3 · The kidney register

**The question for this phase:** where in Nigeria can someone with failed kidneys actually get
dialysis, and when it isn't available, is the problem **equipment, workforce or systems**?

There's no clean public dataset that answers this, so I built one by hand: a **register** with one
row per fact, from one source, with the date, a link and a short quote. Some terms:

- **Haemodialysis** (dialysis): a machine filters the blood when the kidneys can't. Someone with
  **ESKD** (end-stage kidney disease, meaning the kidneys have failed) needs it 2–3 times a week
  for life, or a transplant.
- **Gap type**: why capacity is missing, but only when the source says so:
  - **equipment**: broken or missing machines
  - **workforce**: no nephrologist (kidney specialist) or trained dialysis nurses
  - **systems**: power, water treatment, maintenance, consumables, referral or payment problems
  - **unknown**: the source doesn't say. I never guess.

Every number below is printed by code. None are typed in.

In [1]:
import sys, json
from pathlib import Path
import pandas as pd
import plotly.io as pio

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT / "pipeline"), str(ROOT / "src")]
import charts, load_register

pio.renderers.default = "plotly_mimetype"
pd.set_option("display.max_rows", 80, "display.width", 180, "display.max_colwidth", 120)

REG = ROOT / "data" / "register"
P = ROOT / "data" / "processed"
reg = load_register.load()
by_state = pd.read_csv(P / "register_kidney_by_state.csv")
national = pd.read_csv(P / "register_kidney_national.csv", dtype=str, keep_default_na=False)
panel = pd.read_csv(P / "state_panel.csv")

## 1. How the register was built

1. **Search.** Researchers searched state by state (and nationally) for facts about public dialysis
   capacity: centres, machines, nephrologists, and reported problems. Every fact had to come
   from a page they opened, with an exact quote. Their raw output is kept in `data/register/research/`.
2. **Curate.** `pipeline/curate_register.py` applies a short, written set of rules: drop undated
   rows, move price facts to a separate cost file (for the phase 4 centre model), give each
   hospital one name, and fix the few rows where the recorded status said more than the quote.
3. **Check.** I re-opened key sources and checked the quotes against the live pages (`verification_log.csv`).

In [2]:
research_rows = sum(len(json.loads(f.read_text(encoding="utf-8-sig"))) for f in (REG / "research").glob("*.json"))
cost_rows = len(pd.read_csv(REG / "kidney_cost_evidence.csv"))
vlog = pd.read_csv(REG / "verification_log.csv")
k = reg[reg.specialty == "kidney"]
print(f"Research facts found:        {research_rows}")
print(f"Moved to cost evidence:      {cost_rows}")
print(f"Register rows (kidney):      {len(k)}  ({(k.state == 'National').sum()} national, {(k.state != 'National').sum()} state)")
print(f"States with at least one row: {(by_state.evidence == 'found').sum()} of {len(by_state)}")
print(f"Quotes re-checked on live pages: {len(vlog)}, of which verified: {vlog.verdict.str.startswith('verified').sum()}")

Research facts found:        188
Moved to cost evidence:      7
Register rows (kidney):      178  (24 national, 154 state)
States with at least one row: 37 of 37
Quotes re-checked on live pages: 10, of which verified: 10


In [3]:
print("Sources by type:"); print(k.source_type.value_counts().to_string())
print("\nGap type (as stated by the source):"); print(k.gap_type.value_counts().to_string())
print(f"\n'unknown' gap types: {(k.gap_type == 'unknown').sum()} of {len(k)} rows "
      f"({(k.gap_type == 'unknown').mean():.0%})")
print(f"Oldest source: {k.source_date.min()} · newest: {k.source_date.max()}")

Sources by type:
source_type
news                 132
government            24
peer-reviewed         21
professional body      1

Gap type (as stated by the source):
gap_type
unknown      134
systems       22
equipment     17
workforce      5

'unknown' gap types: 134 of 178 rows (75%)
Oldest source: 2010-12-27 · newest: 2026-11-19


**Read this first:** most rows come from **news** reports, and most sources don't say *why* capacity
is missing. A register like this is a starting map for the "Assess" step, not a replacement for it.

## 2. Nobody agrees how much dialysis Nigeria has

Before looking state by state, the national picture. Published counts of centres, machines and
nephrologists disagree, partly because they count different things (working vs installed) and
come from different years.

In [4]:
charts.national_counts(national).show()

In [5]:
n = national.assign(n=pd.to_numeric(national["count"], errors="coerce"))
centres = n[(n.item == "dialysis centre") & (n.ownership == "mixed")].dropna(subset=["n"])
print("Dialysis centres, as published:")
print(centres[["n", "status", "source_type", "source_date", "source_url"]].sort_values("n").to_string(index=False))

Dialysis centres, as published:
    n  status   source_type source_date                                                                                                                                                                         source_url
 80.0 working peer-reviewed  2022-07-14                                                                                                                                   https://pmc.ncbi.nlm.nih.gov/articles/PMC9528372
 80.0 working          news  2025-08-29                                                             https://nigeriahealthwatch.com/articles/thought-leadership/dialysis-subsidy-in-nigeria-a-lifeline-or-a-short-term-fix/
149.0 unknown    government     2025-12 https://www.gov.uk/government/publications/nigeria-country-policy-and-information-notes/country-information-note-medical-treatment-and-healthcare-nigeria-december-2025-accessible
186.0 unknown peer-reviewed  2022-07-14                                                     

## 3. What kind of gap, state by state

In [6]:
charts.register_gap_types(by_state).show()

In [7]:
s = by_state[by_state.evidence == "found"]
for g in ["equipment", "workforce", "systems"]:
    states = s.loc[s[f"gap_{g}"] > 0, "state"].tolist()
    print(f"{g.capitalize():10} stated in {len(states):2} states: {', '.join(states)}")

Equipment  stated in 11 states: Benue, Edo, Federal Capital Territory, Kano, Katsina, Kwara, Lagos, Osun, Plateau, Taraba, Zamfara
Workforce  stated in  5 states: Akwa Ibom, Bauchi, Benue, Borno, Jigawa
Systems    stated in 17 states: Akwa Ibom, Bauchi, Bayelsa, Benue, Borno, Federal Capital Territory, Gombe, Jigawa, Kano, Kebbi, Kogi, Lagos, Niger, Oyo, Sokoto, Taraba, Yobe


Some examples of each, in the sources' own words:

In [8]:
for g in ["equipment", "workforce", "systems"]:
    ex = k[(k.gap_type == g) & (k.state != "National")].sort_values("source_date", ascending=False).head(3)
    print(f"--- {g.upper()}")
    for _, r in ex.iterrows():
        print(f"{r.state}, {r.facility_name or 'state-wide'} ({r.source_date}, {r.source_type}):")
        print(f"   {r.evidence_note[:260]}")
    print()

--- EQUIPMENT
Edo, University of Benin Teaching Hospital (2026-05, news):
   A patient said only two machines served the subsidy scheme, far too few for demand. This is the patient's claim and the reporter did not verify it. Date is approximate: the page showed '6 months ago'. "They brought just two machines. Two machines for over 400 
Taraba, Federal Medical Centre, Jalingo (2026-01-09, news):
   Reported while the state specialist hospital was shut. The fallback at FMC Jalingo was limited. Quote: "dialysis services at the Federal Medical Centre (FMC), Jalingo" are "constrained by malfunctioning machines and limited capacity."
Taraba, Federal Medical Centre, Jalingo (2026-01, news):
   A second outlet on the same January 2026 shutdown reports FMC Jalingo's machines are out of service. The page carries no exact date; it cites a letter dated 8 Jan 2026. Quote: "the dialysis machines at the Federal Medical Centre (FMC), Jalingo, are currently n

--- WORKFORCE
Bauchi, Abubakar Tafawa Bale

## 4. Working machines per million people

Where a source gives a count of **working** machines, I add it up per state. When two sources
disagree, I show both ends as a range (never an average). These are **floors**: a source names
some hospitals, not all of them.

In [9]:
charts.register_machines_per_million(by_state, panel).show()

In [10]:
cols = ["state", "public_dialysis_working", "machines_working_low", "machines_working_high",
        "machines_not_working_low", "machines_not_working_high", "earliest_source_date", "latest_source_date"]
by_state[cols].sort_values("machines_working_high", ascending=False, na_position="last")

,state,public_dialysis_working,machines_working_low,machines_working_high,machines_not_working_low,machines_not_working_high,earliest_source_date,latest_source_date
24,Lagos,yes,16.0,18.0,24.0,24.0,2019-07-31,2025-08-20
19,Kano,yes,12.0,12.0,14.0,16.0,2017-05-25,2025-10-12
35,Yobe,yes,11.0,11.0,NaN,NaN,2025-03,2026-09-03
12,Ekiti,yes,6.0,6.0,NaN,NaN,2020-01-28,2025-04-22
14,Federal Capital Territory,yes,5.0,5.0,NaN,NaN,2025-01-14,2026
6,Benue,yes,4.0,4.0,2.0,2.0,2015-01-27,2023-11-08
30,Oyo,yes,3.0,3.0,NaN,NaN,2013-04-05,2025-08-20
25,Nasarawa,yes,3.0,3.0,NaN,NaN,2020-01-28,2026-01-29
11,Edo,yes,2.0,2.0,NaN,NaN,2021-05-24,2026-09-24
23,Kwara,yes,2.0,2.0,NaN,NaN,2017-04-05,2021-07-20


## 5. A closer look at Bauchi (Halima's working state)

Bauchi came out of the scorecard as a strong candidate for Halima's story: a big hospital gap and
low ability to pay. Here's everything the register holds for Bauchi.

In [11]:
b = k[k.state == "Bauchi"].sort_values("source_date")
for _, r in b.iterrows():
    print(f"{r.source_date} · {r.facility_name} · {r['item']} · count={r['count'] or '-'} · {r.status} · gap: {r.gap_type}")
    print(f"   {r.evidence_note[:300]}")
    print(f"   {r.source_url}")

2022-02-23 · RCCG-donated dialysis centre, Bauchi (presented at ATBUTH; the article does not state the host facility) · dialysis machine · count=3 · unknown · gap: unknown
   The RCCG donated a dialysis centre to Bauchi State with 3 machines, a reverse-osmosis water system and a 30KVA generator. The handover took place at ATBUTH. Quote: "the center in Bauchi has three Dialysis Machines"
   https://www.withinnigeria.com/news/2022/02/23/bauchi-hails-rccg-on-dialysis-centre-donation/?amp=1
2024-05-09 · Bauchi State Specialist Hospital · dialysis machine · count=4 · unknown · gap: unknown
   The State Executive Council approved buying four dialysis machines for the state specialist hospital, mainly for less-privileged patients. This is an approval, not a confirmed installation. Quote: "four world-class renal dialysis machines"
   https://ait.live/bauchi-govt-approves-construction-of-115-km-of-rural-roads-acquires-renal-dialysis-machines/
2024-05-22 · Bauchi State Specialist Hospital · dial

## 6. Why the register isn't in the scorecard (yet)

The brief says the supply-gap pillar can use register counts. I've held back, because the counts
are too uneven to compare states fairly:

In [12]:
has_count = by_state.machines_working_high.notna().sum()
print(f"States with any count of working machines: {has_count} of {len(by_state)}")
print(f"States whose newest source is older than 2023: "
      f"{(by_state.latest_source_date.astype(str).str[:4].astype(int) < 2023).sum()}")

States with any count of working machines: 12 of 37
States whose newest source is older than 2023: 5


Ranking states on counts that exist for only some of them would reward the states that got press
coverage, not the ones with more machines. The register instead feeds the story (what kind of gap)
and the centre model (phase 4).

## 7. Checks

In [13]:
import validate
errors = validate.check_register()
print("Register checks passed: every row has allowed values, an official state name, a source URL, "
      "a source date and an evidence note." if not errors else errors)

Register checks passed: every row has allowed values, an official state name, a source URL, a source date and an evidence note.
